# 옵티마이저 ( Optimizer )

- 우리의 목표는 손실 함수(loss function)의 값을 최소화하여 가장 정확한 예측을 할 수 있는 모델 파라미터를 찾는것 
- 이 목표를 달성하기 위해 사용하는 도구

- 옵티마이저는 모델이 학습 데이터로부터 얼마나 잘 학습하고 있는지를 평가하는 손실 함수의 값을 최소화 하기 위해 모델 파라미터를 조정하는 방법 

- 가장 기본적인 알고리즘은 `경사 하강법(Gradient Descent)`
- 경사하강법은 손실 함수의 기울기를 계산하여, 이 기울기가 감소하는 방향으로 모델 파라미터를 조금씩 업데이트 


In [ ]:
import torch.optim as optim

아래처럼 많이 쓰는 옵티마이저를 정리할 수 있습니다.

| 이름 | 설명 | PyTorch 코드 |
|---|---|---|
| SGD | 가장 기본적인 경사하강법. 학습률에 민감하지만 단순하고 해석이 쉬움 | optimizer = torch.optim.SGD(model.parameters(), lr=0.01) |
| SGD + Momentum | 이전 기울기 방향을 누적해 진동을 줄이고 수렴을 빠르게 함 | optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9) |
| Nesterov SGD | 모멘텀의 개선형. 미리 한 스텝 앞을 보고 보정 | optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True) |
| Adagrad | 파라미터별로 학습률을 자동 조절. 희소 데이터에 유리 | optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01) |
| RMSprop | Adagrad의 학습률 급감 문제를 완화. RNN 계열에서 자주 사용 | optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001, alpha=0.99) |
| Adam | Momentum + RMSprop 아이디어 결합. 가장 널리 쓰이는 기본 선택 | optimizer = torch.optim.Adam(model.parameters(), lr=0.001) |
| AdamW | Adam의 weight decay를 올바르게 분리. Transformer 계열에서 표준처럼 사용 | optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01) |
| RAdam | Adam 초반 불안정을 보완한 변형 | optimizer = torch.optim.RAdam(model.parameters(), lr=0.001) |
| NAdam | Adam + Nesterov 결합 | optimizer = torch.optim.NAdam(model.parameters(), lr=0.001) |

실무에서 시작 추천:
1. 일반 딥러닝: Adam
2. Transformer/NLP: AdamW
3. 단순 모델 또는 전통적 세팅: SGD + Momentum

In [3]:
import torch.nn as nn
import torch.optim as optim
# 딥러닝 모델 정의
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(in_features=1, out_features=5)  # 첫 번째 선형 레이어
        self.relu = nn.ReLU()  # ReLU 활성화 함수
        self.layer2 = nn.Linear(in_features=5, out_features=1)  # 두 번째 선형 레이어

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# 모델 인스턴스 생성
model = SimpleNN()

# SGD 옵티마이저 설정
optimizer = optim.SGD(model.parameters(), lr=0.01)
print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [4]:
print("Model named parameters:")
for name, param in model.named_parameters():
    print(name, param.size())

Model named parameters:
layer1.weight torch.Size([5, 1])
layer1.bias torch.Size([5])
layer2.weight torch.Size([1, 5])
layer2.bias torch.Size([1])


In [ ]:
# __setattr__ 예제 1) 일반 Python 클래스
class LoggerObject:
    def __setattr__(self, name, value):
        print(f"[LoggerObject] setting {name} = {value}")
        super().__setattr__(name, value)

obj = LoggerObject()
obj.x = 10
obj.message = "hello"
print("obj.__dict__:", obj.__dict__)


# __setattr__ 예제 2) nn.Module 내부 동작 확인
# nn.Module은 __setattr__를 오버라이드하여 nn.Parameter를 자동 등록한다.
import torch
import torch.nn as nn

class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(3, 3))  # 자동으로 파라미터 등록
        self.bias = torch.randn(3)                      # 일반 Tensor는 파라미터 미등록

m = MyModule()

print("\n[named_parameters()]")
for name, p in m.named_parameters():
    print(name, p.shape)

print("\n[속성 타입 확인]")
print("type(m.weight):", type(m.weight))
print("type(m.bias):", type(m.bias))
print("'weight' in dict(named_parameters):", "weight" in dict(m.named_parameters()))
print("'bias' in dict(named_parameters):", "bias" in dict(m.named_parameters()))